# 🐦 Twitter/X Scraper — Indonesian Government Law Sentiment
**Method:** `twscrape` (no API key needed, uses regular Twitter account)

**What this collects:**
- Original tweets
- Retweets
- Replies
- Metadata: likes, retweet count, language, user info, date

> ⚠️ You need a regular Twitter/X account (free). No developer account needed.

## ⚙️ Step 1 — Install Dependencies

In [ ]:
!pip install twscrape pandas openpyxl nest_asyncio -q
print('✅ All packages installed!')

## 🔐 Step 2 — Add Your Twitter Account

> Fill in your Twitter/X credentials below. It's recommended to use a **secondary/burner account** just in case.
> Your credentials are only used locally in this Colab session.

In [ ]:
import twscrape
import nest_asyncio
import asyncio
nest_asyncio.apply()

# ============================================================
# 🔐 FILL IN YOUR CREDENTIALS HERE
# ============================================================
TWITTER_USERNAME  = "your_username"       # without @
TWITTER_PASSWORD  = "your_password"
TWITTER_EMAIL     = "your_email@gmail.com"
# ============================================================

async def setup_account():
    api = twscrape.API()
    await api.pool.add_account(
        username=TWITTER_USERNAME,
        password=TWITTER_PASSWORD,
        email=TWITTER_EMAIL,
        email_password=TWITTER_PASSWORD  # same as Twitter password usually
    )
    await api.pool.login_all()
    print('✅ Account logged in successfully!')
    return api

api = asyncio.run(setup_account())

## 🔍 Step 3 — Configure Your Search Topics

Pre-loaded with **Indonesian social media & children law** keywords.
You can add/remove keywords as needed.

In [ ]:
# ============================================================
# 🔍 SEARCH CONFIGURATION
# ============================================================

SEARCH_QUERIES = [
    # --- Indonesian keywords ---
    "larangan medsos anak",
    "UU media sosial anak",
    "ban anak sosmed",
    "perlindungan anak media sosial",
    "RUU anak digital",
    "#UUPerlindunganAnak",
    "anak dilarang medsos",
    "regulasi sosmed anak Indonesia",

    # --- English keywords (for international coverage) ---
    "Indonesia children social media ban",
    "Indonesia social media law children",
]

MAX_TWEETS_PER_QUERY = 50   # 50 × 10 queries = up to 500 tweets total
INCLUDE_REPLIES = True      # Set False to skip replies

print(f'📋 {len(SEARCH_QUERIES)} queries loaded')
print(f'🎯 Target: up to {MAX_TWEETS_PER_QUERY * len(SEARCH_QUERIES)} tweets total')

## 🚀 Step 4 — Run the Scraper

In [ ]:
import pandas as pd
from datetime import datetime

all_tweets = []
seen_ids = set()  # deduplication

def parse_tweet(tweet, query, tweet_type="tweet"):
    """Extract relevant fields from a tweet object."""
    return {
        # --- Identity ---
        "tweet_id":          str(tweet.id),
        "tweet_type":        tweet_type,
        "query_used":        query,

        # --- Content ---
        "text":              tweet.rawContent,
        "language":          tweet.lang,
        "created_at":        tweet.date.strftime("%Y-%m-%d %H:%M:%S"),
        "url":               tweet.url,

        # --- Engagement ---
        "like_count":        tweet.likeCount,
        "retweet_count":     tweet.retweetCount,
        "reply_count":       tweet.replyCount,
        "quote_count":       tweet.quoteCount,

        # --- User ---
        "user_id":           str(tweet.user.id),
        "username":          tweet.user.username,
        "display_name":      tweet.user.displayname,
        "user_followers":    tweet.user.followersCount,
        "user_verified":     tweet.user.verified,
        "user_location":     tweet.user.location,

        # --- Reply/Thread context ---
        "in_reply_to_id":    str(tweet.inReplyToTweetId) if tweet.inReplyToTweetId else None,
        "in_reply_to_user":  tweet.inReplyToUser.username if tweet.inReplyToUser else None,

        # --- Retweet context ---
        "is_retweet":        tweet.retweetedTweet is not None,
        "original_tweet_id": str(tweet.retweetedTweet.id) if tweet.retweetedTweet else None,
        "original_user":     tweet.retweetedTweet.user.username if tweet.retweetedTweet else None,
    }


async def scrape_all():
    for query in SEARCH_QUERIES:
        print(f'\n🔍 Searching: "{query}"')
        count = 0
        try:
            # Build search string
            search_str = query
            if INCLUDE_REPLIES:
                # twscrape includes replies by default with 'include:replies'
                search_str = f"{query} include:replies"

            async for tweet in api.search(search_str, limit=MAX_TWEETS_PER_QUERY):
                if str(tweet.id) in seen_ids:
                    continue
                seen_ids.add(str(tweet.id))

                # Determine tweet type
                if tweet.retweetedTweet:
                    t_type = "retweet"
                elif tweet.inReplyToTweetId:
                    t_type = "reply"
                else:
                    t_type = "original"

                all_tweets.append(parse_tweet(tweet, query, t_type))
                count += 1

                if count % 10 == 0:
                    print(f'  → {count} tweets collected...')

        except Exception as e:
            print(f'  ⚠️ Error on query "{query}": {e}')

        print(f'  ✅ {count} tweets from this query')

asyncio.run(scrape_all())
print(f'\n🎉 DONE! Total unique tweets collected: {len(all_tweets)}')

## 📊 Step 5 — Preview & Basic Stats

In [ ]:
df = pd.DataFrame(all_tweets)

print('=== 📊 DATASET SUMMARY ===')
print(f'Total tweets      : {len(df)}')
print(f'Unique users      : {df["user_id"].nunique()}')
print(f'Date range        : {df["created_at"].min()} → {df["created_at"].max()}')
print()
print('--- Tweet Types ---')
print(df['tweet_type'].value_counts().to_string())
print()
print('--- Languages ---')
print(df['language'].value_counts().to_string())
print()
print('--- Top Queries by Volume ---')
print(df['query_used'].value_counts().to_string())

df.head(5)

## 💾 Step 6 — Save to Files

In [ ]:
from google.colab import files
import os

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# --- Save CSV (main format for sentiment analysis) ---
csv_filename = f'tweets_indonesia_law_{timestamp}.csv'
df.to_csv(csv_filename, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel compatibility
print(f'✅ Saved CSV: {csv_filename}')

# --- Save Excel (with sheet breakdown) ---
xlsx_filename = f'tweets_indonesia_law_{timestamp}.xlsx'
with pd.ExcelWriter(xlsx_filename, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='All Tweets', index=False)
    df[df['tweet_type'] == 'original'].to_excel(writer, sheet_name='Original Tweets', index=False)
    df[df['tweet_type'] == 'retweet'].to_excel(writer, sheet_name='Retweets', index=False)
    df[df['tweet_type'] == 'reply'].to_excel(writer, sheet_name='Replies', index=False)
print(f'✅ Saved Excel: {xlsx_filename}')

# --- Auto-download ---
print('\n📥 Downloading files...')
files.download(csv_filename)
files.download(xlsx_filename)

## 🔁 Optional: Scrape Replies to a Specific Tweet
If you find an important tweet and want all its replies:

In [ ]:
# ============================================================
# Paste a tweet URL or ID to get all its replies
# ============================================================
TWEET_ID = "PASTE_TWEET_ID_HERE"  # e.g. "1234567890123456789"

reply_tweets = []

async def get_replies():
    async for tweet in api.search(f"conversation_id:{TWEET_ID}", limit=200):
        reply_tweets.append(parse_tweet(tweet, f"conversation:{TWEET_ID}", "reply"))
    print(f'✅ {len(reply_tweets)} replies collected')

asyncio.run(get_replies())

df_replies = pd.DataFrame(reply_tweets)
df_replies.head()

---
## 📌 Notes & Tips

| Field | Use for Sentiment Analysis |
|---|---|
| `text` | Main text to analyze |
| `language` | Filter `id` (Indonesian) vs `en` (English) |
| `tweet_type` | Separate original vs reply vs retweet |
| `like_count` / `retweet_count` | Weight tweets by influence |
| `user_followers` | Identify influencers |
| `created_at` | Temporal trend analysis |

**Recommended next steps:**
1. Clean text (remove URLs, @mentions, extra spaces)
2. Use `IndoNLU` or `indobert` for Indonesian sentiment
3. Use `transformers` (multilingual BERT) for mixed language
4. Visualize with `matplotlib` or `wordcloud`